In [0]:
%run ../01_Bronze/00_setup

In [0]:
# process new records into the Silver layer bin table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_bins(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.bins"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("bin_id")
    
    updates_df = microBatchDF.join(existing_ids, "bin_id", "inner") \
        .withColumn("merge_key", F.col("bin_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    combined_df.createOrReplaceTempView("source_bins_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_bins_view AS source
        ON target.bin_id = source.merge_key AND target.is_active = true
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND target.status <> source.status THEN
          UPDATE SET target.is_active = false, target.end_date = current_timestamp(), target.updated_timestamp = current_timestamp()
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (bins_sk, bin_id, order_id, item_count, status, current_weight, is_active, start_date, end_date, updated_timestamp)
          VALUES (md5(concat(source.bin_id, cast(source.updated_timestamp as STRING))), 
                  source.bin_id, source.order_id, source.item_count, source.status, source.current_weight, 
                  true, source.updated_timestamp, NULL, current_timestamp())
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.bin_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_bins)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_bin_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())